# Multi-Agent Ticket Triage System

This notebook demonstrates how to build a **multi-agent system** using Azure AI Agent Service for automated ticket triage. The system uses multiple specialized agents that work together to analyze support tickets and determine:

- **Priority Level** (High/Medium/Low)
- **Team Assignment** (Frontend/Backend/Infrastructure/Marketing)
- **Effort Estimation** (Small/Medium/Large)

## Architecture Overview

The system consists of:
1. **Three Specialist Agents**: Each focused on one aspect of ticket analysis
2. **One Orchestrator Agent**: Uses the specialist agents as tools to provide comprehensive triage
3. **Connected Agent Tools**: Enable the orchestrator to call specialist agents as needed

## 📦 Import Required Libraries and Setup Environment

This cell imports all the necessary libraries for building our multi-agent system and loads environment variables from the `.env` file. We need:

- **Azure AI Agents SDK**: To create and manage multiple AI agents
- **Azure Identity**: For authentication with Azure services
- **Environment variables**: Project endpoint and model deployment details

In [ ]:
import os
from pathlib import Path
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import ConnectedAgentTool, MessageRole, ListSortOrder
from azure.identity import InteractiveBrowserCredential
from dotenv import load_dotenv

# Load environment variables from parent .env
notebook_path = Path().absolute()
parent_dir = notebook_path.parent
load_dotenv(parent_dir / '.env')

# Get tenant ID and project endpoint - fix the environment variable names
tenant_id = os.environ.get("TENANT_ID")
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment = os.getenv("MODEL_DEPLOYMENT_NAME")

print(f"🔑 Using Tenant ID: {tenant_id}")
print(f"🔗 Project Endpoint: {project_endpoint}")
print(f"🤖 Model Deployment: {model_deployment}")

# Verify we have all required environment variables
if not project_endpoint:
    print("❌ Error: AI_FOUNDRY_PROJECT_ENDPOINT not found in environment variables")
    print("💡 Make sure your .env file contains AI_FOUNDRY_PROJECT_ENDPOINT")
elif not model_deployment:
    print("❌ Error: MODEL_DEPLOYMENT_NAME not found in environment variables") 
    print("💡 Make sure your .env file contains MODEL_DEPLOYMENT_NAME")
else:
    print("✅ All required environment variables found")

## 🎯 Define Specialist Agent Instructions

Now we'll define the instructions for each of our three specialist agents. Each agent has a specific role in the ticket triage process.

### Priority Assessment Agent

This agent analyzes tickets to determine their urgency level. It categorizes tickets as High, Medium, or Low priority based on their impact on users and business operations.

In [ ]:
# Priority agent definition
priority_agent_name = "priority_agent"
priority_agent_instructions = """
Assess how urgent a ticket is based on its description.

Respond with one of the following levels:
- High: User-facing or blocking issues
- Medium: Time-sensitive but not breaking anything
- Low: Cosmetic or non-urgent tasks

Only output the urgency level and a very brief explanation.
"""

### Team Assignment Agent

This agent determines which team should handle each ticket based on the technical domain and expertise required. It assigns tickets to Frontend, Backend, Infrastructure, or Marketing teams.

In [ ]:
# Team agent definition
team_agent_name = "team_agent"
team_agent_instructions = """
Decide which team should own each ticket.

Choose from the following teams:
- Frontend
- Backend
- Infrastructure
- Marketing

Base your answer on the content of the ticket. Respond with the team name and a very brief explanation.
"""

### Effort Estimation Agent

This agent estimates the amount of work required to resolve each ticket. It categorizes the effort as Small (1 day), Medium (2-3 days), or Large (multi-day/cross-team effort).

In [ ]:
# Effort agent definition
effort_agent_name = "effort_agent"
effort_agent_instructions = """
Estimate how much work each ticket will require.

Use the following scale:
- Small: Can be completed in a day
- Medium: 2-3 days of work
- Large: Multi-day or cross-team effort

Base your estimate on the complexity implied by the ticket. Respond with the effort level and a brief justification.
"""

### Orchestrator Agent Instructions

The main orchestrator agent coordinates all three specialist agents. It receives the ticket and uses the specialist agents as tools to provide a comprehensive triage analysis.

In [ ]:
# Instructions for the primary agent
triage_agent_instructions = """
Triage the given ticket. Use the connected tools to determine the ticket's priority, 
which team it should be assigned to, and how much effort it may take.
"""

## 🔗 Connect to Azure AI Agent Service

This cell establishes a connection to the Azure AI Agent Service using our project endpoint and credentials. This client will be used to create and manage all our agents.

In [ ]:
# Connect to the agents client using InteractiveBrowserCredential for consistency
credential = InteractiveBrowserCredential(tenant_id=tenant_id)
print("🌐 Launching interactive browser authentication (multi-agent notebook)...")
agents_client = AgentsClient(
    endpoint=project_endpoint,
    credential=credential
)
print("✅ AgentsClient initialized with InteractiveBrowserCredential")

## 🤖 Create Multi-Agent System

This cell creates all four agents in our system:

1. **Three Specialist Agents**: Priority, Team, and Effort assessment agents
2. **Connected Agent Tools**: Wrapper tools that allow the orchestrator to call specialists
3. **Main Orchestrator Agent**: Uses the specialist agents as tools for comprehensive ticket triage

Each specialist agent is created with its specific instructions and then wrapped in a `ConnectedAgentTool` so the orchestrator can use them.

In [ ]:
# FIXED: Create agents without context manager to keep client active
print("Creating agents...")

# Create the priority agent on the Azure AI agent service
priority_agent = agents_client.create_agent(
    model=model_deployment,
    name=priority_agent_name,
    instructions=priority_agent_instructions
)
print("✅ Priority agent created")

# Create a connected agent tool for the priority agent
priority_agent_tool = ConnectedAgentTool(
    id=priority_agent.id, 
    name=priority_agent_name, 
    description="Assess the priority of a ticket"
)

# Create the team agent and connected tool
team_agent = agents_client.create_agent(
    model=model_deployment,
    name=team_agent_name,
    instructions=team_agent_instructions
)
print("✅ Team agent created")

team_agent_tool = ConnectedAgentTool(
    id=team_agent.id, 
    name=team_agent_name, 
    description="Determines which team should take the ticket"
)

# Create the effort agent and connected tool
effort_agent = agents_client.create_agent(
    model=model_deployment,
    name=effort_agent_name,
    instructions=effort_agent_instructions
)
print("✅ Effort agent created")

effort_agent_tool = ConnectedAgentTool(
    id=effort_agent.id, 
    name=effort_agent_name, 
    description="Determines the effort required to complete the ticket"
)

# Create a main agent with the Connected Agent tools
agent = agents_client.create_agent(
    model=model_deployment,
    name="triage-agent",
    instructions=triage_agent_instructions,
    tools=[
        priority_agent_tool.definitions[0],
        team_agent_tool.definitions[0],
        effort_agent_tool.definitions[0]
    ]
)
print("✅ Main triage agent created with connected tools")
print("🎯 All agents are ready for use!")

## 🎯 Execute Multi-Agent Ticket Triage

Now we'll run our multi-agent ticket triage system 

1. **Create a thread**: Automatically creates a new conversation thread

2. **Send the ticket**: Submits the support ticket as the initial message4. **Return results**: Provides the complete triage analysis
3. **Process with agents**: The orchestrator calls each specialist agent for their assessment

In [ ]:
# Define the ticket to triage
prompt = "Users can't reset their password from the mobile app."

## 📊 View Multi-Agent Conversation

After processing, we'll retrieve and display all messages from the conversation thread to see how the orchestrator coordinated with the specialist agents.

In [ ]:
# Modern atomic pattern: create thread, send message, and process in one operation
print(f"🔄 Processing ticket: '{prompt}'")
print("   Creating thread, coordinating agents, and generating triage...\n")

run = agents_client.create_thread_and_process_run(
    agent_id=agent.id,
    thread={
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }
)

print(f"✅ Run finished with status: {run.status}")
print(f"📋 Thread ID: {run.thread_id}\n")

# Check if the run failed
if run.status == "failed":
    print(f"❌ Run failed: {run.last_error}")
else:
    # Fetch run steps to see which agents were called
    print("🔍 Agent Coordination Details:")
    print("=" * 50)
    
    try:
        run_steps = agents_client.run_steps.list(thread_id=run.thread_id, run_id=run.id)
        
        for step in run_steps:
            # Access step_details as dictionary
            step_details = step.get("step_details", {})
            tool_calls = step_details.get("tool_calls", [])
            
            if tool_calls:
                for tool_call in tool_calls:
                    # Check for connected_agent type tool calls
                    if tool_call.get("type") == "connected_agent":
                        connected_agent = tool_call.get("connected_agent", {})
                        agent_name = connected_agent.get("name", "Unknown")
                        print(f"\n🤖 Called Agent: {agent_name}")
                        
                        # Try to get the output if available
                        output = connected_agent.get("output")
                        if output:
                            print(f"   Response: {output}")
    except Exception as e:
        print(f"Note: Could not retrieve run steps: {e}")
    
    print("\n" + "=" * 50)
    
    # Fetch and display all messages from the conversation
    messages = agents_client.messages.list(thread_id=run.thread_id, order=ListSortOrder.ASCENDING)
    
    print("\n📊 Final Conversation:")
    print("=" * 50)
    
    for msg in messages:
        if msg.text_messages:
            last_text = msg.text_messages[-1]
            print(f"\n🤖 {msg.role.upper()}:")
            print(f"{last_text.text.value}")
    
    print("\n" + "=" * 50)
    print("🎯 Multi-agent ticket triage completed!")

## 🧹 Clean Up Resources

This cell deletes all the agents we created to avoid leaving resources running in Azure. It's important to clean up agents after use to prevent unnecessary costs.

In [ ]:
from azure.core.exceptions import ResourceNotFoundError

print("🧹 Cleaning up agents...\n")

# Delete the main triage agent
try:
    agents_client.delete_agent(agent.id)
    print("🗑️  Deleted triage agent")
except ResourceNotFoundError:
    print("ℹ️   Triage agent already deleted or not found")

# Delete the specialist agents
for agent_obj, agent_name in [(priority_agent, "priority"), (team_agent, "team"), (effort_agent, "effort")]:
    try:
        agents_client.delete_agent(agent_obj.id)
        print(f"🗑️  Deleted {agent_name} agent")
    except ResourceNotFoundError:
        print(f"ℹ️   {agent_name.capitalize()} agent already deleted or not found")

print("\n✅ Cleanup completed!")